In [3]:
# 📦 Install if not already done
# !pip install langchain openai faiss-cpu tiktoken python-dotenv

# 1️⃣ Imports
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
from langchain.chains import RetrievalQA
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# GROQ & tavily API Key
GROQ_API_KEY = ""


# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=300,
)

# 4️⃣ Create Vector DB (Retriever)

with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
from langchain.text_splitter import CharacterTextSplitter
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 5️⃣ Create RetrievalQA Chain
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

# 6️⃣ Convert to LangChain Tool
qa_tool = Tool(
    name="LangChainRetriever",
    func=qa_chain.run,
    description="Use this to answer questions about LangChain framework, features, or its creator."
)

# 7️⃣ Setup Memory (to support conversational context)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 8️⃣ Initialize Agent with Tool + Memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,  # ReAct agent
    memory=memory,
    verbose=True,
    handle_parsing_errors=True
)

# 9️⃣ Ask Questions (RAG-Style)
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.run("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Combined Reasoning")
res3 = agent.run("Explain LangChain's use in AI workflows.")
print("Answer:", res3)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1567.11it/s]
C:\Users\PAVAN\AppData\Local\Temp\ipykernel_16104\4186855812.py:53: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(
C:\Users\PAVAN\AppData\Local\Temp\ipykernel_16104\4186855812.py:64: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  res1 = agent.run("What is LangChain?")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


1️⃣ First Question
Thought: Do I need to use a tool? Yes
Action: LangChainRetriever
Action Input: What is LangChain?
Observation: LangChain is a framework for building applications with Large Language Models (LLMs). It was created by Harrison Chase and supports various features such as RAG, agents, memory, tools, and more. It's commonly used in applications like chatbots, document Q&A, and AI workflows.
Thought:

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Do I need to use a tool? No
AI: LangChain is a framework for building applications with Large Language Models (LLMs). It was created by Harrison Chase and supports various features such as RAG, agents, memory, tools, and more. It's commonly used in applications like chatbots, document Q&A, and AI workflows.

> Finished chain.
Answer: LangChain is a framework for building applications with Large Language Models (LLMs). It was created by Harrison Chase and supports various features such as RAG, agents, memory, tools, and more. It's commonly used in applications like chatbots, document Q&A, and AI workflows.

2️⃣ Follow-up
Thought: Do I need to use a tool? Yes
Action: LangChainRetriever
Action Input: Who created LangChain?
Observation: Harrison Chase created LangChain.
Thought:

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Do I need to use a tool? No
AI: Harrison Chase created LangChain.

> Finished chain.
Answer: Harrison Chase created LangChain.

3️⃣ Combined Reasoning
Thought: Do I need to use a tool? No
AI: LangChain is utilized in AI workflows to streamline and automate various tasks. Its features, such as RAG (Retrieve, Augment, Generate) and agents, enable the creation of complex workflows that can interact with large language models. For instance, LangChain can be used to build workflows that involve document processing, question answering, and text generation. Additionally, its support for tools and memory allows for the integration of external services and the storage of relevant data, making it a powerful tool for building AI-driven workflows. Overall, LangChain's use in AI workflows enables developers to create more efficient, automated, and scalable solutions.

> Finished chain.
Answer: LangChain is utilized in AI workflows to streamline and automate various tasks. Its features, such as RAG 